# BASE Fellows Coding Test

Welcome! This is the **BASE Fellows coding test**. It is designed to check practical familiarity with Python, PyTorch, and a few foundational AI/LLM concepts.

## Time expectation

Please spend **around 60 minutes** on this notebook. An experienced developer may finish in about 45 minutes; candidates newer to PyTorch may take closer to 90 minutes. **You do not need to get every test passing** to submit. We expect a distribution of scores, and the test is calibrated so that the median score should be around **75%**.

## Public tests and review

The tests in this notebook are public feedback checks. They are here to help you understand the expected behavior, debug your solutions, and estimate your progress.

They are not the entire basis for review. After you submit, reviewers may run additional private checks for edge cases and may also read your code for correctness, clarity, and reasonable use of Python/PyTorch.

You do **not** need to get 100% of the public tests passing to submit. Please submit your best work when you reach the suggested time limit or feel ready; partial solutions still give us useful signal.

Please do not edit the test cells or `fellowship_coding_test_tests.py`. Changing the public tests may change your local output, but it will not affect the reviewer’s private checks.

## Scoring guide

The public tests in this notebook award **100 points** total for your feedback:

| Question | Topic | Points |
| --- | --- | ---: |
| 1 | Python classes and data structures | 15 |
| 2 | Python collections, sorting, edge cases | 15 |
| 3 | Basic PyTorch tensors and modules | 20 |
| 4 | Attention masks and numerically safe softmax | 25 |
| 5 | Temperature scaling and top-p / nucleus sampling probabilities | 25 |

Approximate score bands:

- **90–100**: strong readiness; comfortable with Python, PyTorch, and core AI implementation details.
- **75–89**: ready for the fellowship; may need occasional support on edge cases or tensor details.
- **60–74**: promising foundation; should expect to spend extra time on prep and debugging.
- **Below 60**: likely needs more Python/PyTorch practice before the fellowship pace feels comfortable.

## Instructions

1. In Colab, click **File → Save a copy in Drive** before editing.
2. Run the setup cell below once.
3. For each question, read the prompt, fill in the code stub immediately below it, then run the test cell immediately after the stub.
4. When finished, run the final check and submit the share link requested by the application form.

You may use the Python standard library and PyTorch. You may add helper functions inside the solution cells if useful.


In [ ]:
# Load the external public tests and common imports.
# If this notebook is opened directly in Colab, the sibling .py file may not
# be present, so this cell downloads it from the GitHub repo.
from collections import Counter
from typing import Callable, Dict, Iterable, List, Optional, Tuple
import importlib
import math
import os
import urllib.request

import torch
from torch import nn

TESTS_FILE = 'fellowship_coding_test_tests.py'
TESTS_URL = 'https://raw.githubusercontent.com/retroam/base_arena/main/fellowship_coding_test_tests.py'

if not os.path.exists(TESTS_FILE):
    urllib.request.urlretrieve(TESTS_URL, TESTS_FILE)

import fellowship_coding_test_tests
importlib.reload(fellowship_coding_test_tests)
print('Loaded external tests from', TESTS_FILE)


## Question 1 — Integer container (15 points)

Implement two operations for adding and removing numbers from the container. Initially, the container is empty.

- `add(self, value: int) -> int` — add the specified integer value to the container and return the total number of stored values after the addition.
- `delete(self, value: int) -> bool` — remove one copy of `value` if present. Return `True` if a value was removed, otherwise return `False`.

Duplicates are allowed.

### Example

```python
container = IntegerContainerImpl()
container.add(5)      # returns 1; state contains [5]
container.add(10)     # returns 2; state contains [5, 10]
container.add(5)      # returns 3; state contains [5, 10, 5]
container.delete(10)  # returns True; state contains [5, 5]
container.delete(1)   # returns False; state contains [5, 5]
```


In [ ]:
class IntegerContainerImpl:
    def __init__(self):
        # TODO: choose whatever internal representation you like.
        pass

    def add(self, value: int) -> int:
        # TODO: implement.
        raise NotImplementedError

    def delete(self, value: int) -> bool:
        # TODO: implement.
        raise NotImplementedError


In [ ]:
from fellowship_coding_test_tests import run_integer_container_tests
run_integer_container_tests()


## Question 2 — Top-k frequent words (15 points)

Implement `top_k_frequent_words(words, k)`. Return the `k` most frequent unique words.

Sorting rules:

1. Higher frequency comes first.
2. Ties are broken alphabetically in ascending order.
3. If `k` is larger than the number of unique words, return all unique words according to the same ordering rules.

Examples:

```python
top_k_frequent_words(['arena', 'base', 'arena'], 2)  # ['arena', 'base']
top_k_frequent_words(['b', 'a'], 2)                  # ['a', 'b']
```


In [ ]:
def top_k_frequent_words(words: List[str], k: int) -> List[str]:
    # TODO: implement.
    raise NotImplementedError


In [ ]:
from fellowship_coding_test_tests import run_top_k_frequent_tests
run_top_k_frequent_tests()


## Question 3 — Basic PyTorch (20 points)

Implement two small PyTorch functions.

1. For a 2D tensor `x`, `normalize_rows(x, eps=1e-8)` should return a tensor where each row of `x` is divided by its L2 norm. Rows with norm 0 should remain all zeros. Do not mutate `x`, and preserve gradients.
2. `make_tiny_mlp(input_dim, hidden_dim, output_dim)` should return a neural network with architecture:

```text
Linear(input_dim, hidden_dim) -> ReLU -> Linear(hidden_dim, output_dim)
```


In [ ]:
def normalize_rows(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    # TODO: implement.
    raise NotImplementedError


def make_tiny_mlp(input_dim: int, hidden_dim: int, output_dim: int) -> nn.Module:
    # TODO: implement.
    raise NotImplementedError


In [ ]:
from fellowship_coding_test_tests import run_pytorch_basics_tests
run_pytorch_basics_tests()


## Question 4 — Masked attention softmax (25 points)

In transformer attention, a model computes attention scores and then applies a mask before softmax. Implement `masked_softmax(scores, mask, dim=-1)`.

Arguments:

- `scores`: a floating point PyTorch tensor of any shape.
- `mask`: a boolean tensor that is broadcastable to `scores`. Positions where `mask` is `True` are valid; positions where `mask` is `False` should receive probability `0`.
- `dim`: the dimension over which to normalize.

Requirements:

- Return probabilities with the same shape as `scores`.
- Valid positions in each slice along `dim` should sum to `1`.
- Masked positions should be exactly `0`.
- If every position in a slice along `dim` is masked out, return all zeros for that slice instead of `NaN`.
- Do not mutate `scores`, and preserve gradients for valid positions.

Example:

```python
scores = torch.tensor([[1.0, 2.0, 3.0]])
mask = torch.tensor([[True, False, True]])
masked_softmax(scores, mask)
# approximately tensor([[0.1192, 0.0000, 0.8808]])
```


In [ ]:
def masked_softmax(scores: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    # TODO: implement.
    raise NotImplementedError


In [ ]:
from fellowship_coding_test_tests import run_masked_softmax_tests
run_masked_softmax_tests()


## Question 5 — Temperature and top-p probabilities (25 points)

Many language models turn logits into sampling probabilities by applying temperature scaling and then top-p (nucleus) filtering.

Implement `top_p_probs(logits, top_p=0.9, temperature=1.0)`.

Definitions:

- `logits` is a 1D PyTorch tensor of unnormalized scores for the next token.
- Temperature scaling divides logits by `temperature` before softmax. Lower temperatures make the distribution sharper; higher temperatures make it flatter.
- Top-p filtering keeps the smallest set of highest-probability tokens whose cumulative probability is at least `top_p`, then renormalizes over only those tokens.

Requirements:

- Return a 1D probability tensor with the same shape as `logits`.
- Output probabilities should sum to `1`.
- Tokens outside the top-p nucleus should have probability exactly `0`.
- Always keep at least one token, even for very small `top_p`.
- Preserve the original token order in the returned tensor.
- Raise `ValueError` if `temperature <= 0` or if `top_p` is not in `(0, 1]`.

Example:

```python
logits = torch.tensor([4.0, 3.0, 1.0, 0.0])
top_p_probs(logits, top_p=0.8, temperature=1.0)
# keeps the highest-probability tokens needed to reach 0.8 cumulative probability,
# zeros out the rest, and renormalizes.
```


In [ ]:
def top_p_probs(logits: torch.Tensor, top_p: float = 0.9, temperature: float = 1.0) -> torch.Tensor:
    # TODO: implement.
    raise NotImplementedError


In [ ]:
from fellowship_coding_test_tests import run_top_p_probs_tests
run_top_p_probs_tests()


## Final check

Run this cell before submitting your notebook link. It prints your public-test point total and score band.

The public score is a helpful progress indicator, not the only review criterion. Reviewers may run additional private checks and may inspect your code manually. Submit your notebook even if some public tests are still failing, especially if you have reached the suggested time limit.


In [ ]:
from fellowship_coding_test_tests import run_all_tests
run_all_tests()
